# Bangladesh Sleep Duration Model Analysis

This notebook tests whether the Bangladesh Sleep Patterns dataset can replace the weak NHANES sleep-duration model.

The goal is to model:

caffeine + exercise + lifestyle factors → predicted sleep duration

The notebook first tries to download the dataset automatically from Mendeley Data.

If the automatic download works, the dataset is saved inside the project data folder and loaded directly.

If the automatic download fails, the notebook will show a clear manual fallback message.

In [3]:
# ------------------------------------------------------------
# Step 1: Download / find / load Bangladesh Sleep Patterns dataset
# ------------------------------------------------------------

from pathlib import Path
import zipfile
import shutil
import subprocess
import sys
from urllib.parse import urljoin

# ------------------------------------------------------------
# Import needed packages
# ------------------------------------------------------------

try:
    import requests
except ImportError:
    print("requests is not installed yet. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests

try:
    from bs4 import BeautifulSoup
except ImportError:
    print("beautifulsoup4 is not installed yet. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "beautifulsoup4"])
    from bs4 import BeautifulSoup

import pandas as pd


# ------------------------------------------------------------
# Re-create paths in case PyCharm/Jupyter forgets them
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external_sleep_datasets"
BANGLADESH_DATA_DIR = EXTERNAL_DATA_DIR / "bangladesh_sleep_patterns"

EXTERNAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
BANGLADESH_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Bangladesh dataset folder:")
print(BANGLADESH_DATA_DIR)


# ------------------------------------------------------------
# Dataset information
# ------------------------------------------------------------

DATASET_PAGE_URL = "https://data.mendeley.com/datasets/y74tv2jbz3/1"
ZIP_PATH = BANGLADESH_DATA_DIR / "bangladesh_sleep_patterns.zip"


# ------------------------------------------------------------
# Helper function: find existing CSV files
# ------------------------------------------------------------

def find_csv_files(folder):
    """
    Find all CSV files inside a folder and its subfolders.
    """
    return sorted(folder.rglob("*.csv"))


# ------------------------------------------------------------
# Helper function: find existing ZIP files
# ------------------------------------------------------------

def find_zip_files(folder):
    """
    Find all ZIP files inside a folder and its subfolders.
    """
    return sorted(folder.rglob("*.zip"))


# ------------------------------------------------------------
# Helper function: extract ZIP files
# ------------------------------------------------------------

def extract_zip_file(zip_path, output_folder):
    """
    Extract a ZIP file into the selected output folder.
    """

    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"This file is not a valid ZIP file: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_folder)

    print(f"Extracted ZIP file: {zip_path.name}")


# ------------------------------------------------------------
# Helper function: try automatic Mendeley download
# ------------------------------------------------------------

def try_mendeley_download():
    """
    Try to download the Mendeley dataset automatically.

    This may fail with 403 Forbidden because Mendeley sometimes blocks
    notebook-based downloads.
    """

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }

    response = requests.get(
        DATASET_PAGE_URL,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    download_url = None

    for link in soup.find_all("a", href=True):
        link_text = link.get_text(" ", strip=True).lower()
        href = link["href"]

        if "download all" in link_text:
            download_url = urljoin(DATASET_PAGE_URL, href)
            break

        if "file_downloaded" in href:
            download_url = urljoin(DATASET_PAGE_URL, href)
            break

        if href.lower().endswith(".zip"):
            download_url = urljoin(DATASET_PAGE_URL, href)
            break

    if download_url is None:
        raise ValueError("Could not find a download link on the Mendeley page.")

    print("Download link found:")
    print(download_url)

    with requests.get(
        download_url,
        headers=headers,
        stream=True,
        timeout=60,
        allow_redirects=True
    ) as download_response:
        download_response.raise_for_status()

        with open(ZIP_PATH, "wb") as file:
            for chunk in download_response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)

    if not zipfile.is_zipfile(ZIP_PATH):
        raise ValueError(
            "Downloaded file was not a valid ZIP file. "
            "Mendeley may have returned a blocked HTML page."
        )

    extract_zip_file(ZIP_PATH, BANGLADESH_DATA_DIR)


# ------------------------------------------------------------
# Helper function: search Windows Downloads folder
# ------------------------------------------------------------

def search_downloads_folder():
    """
    Search the user's Downloads folder for the Bangladesh dataset ZIP or CSV.

    This is useful when Mendeley blocks automatic downloading.
    """

    possible_download_folders = [
        Path.home() / "Downloads",
        Path("D:/Downloads"),
        Path("C:/Users") / Path.home().name / "Downloads",
    ]

    search_terms = [
        "bangladesh",
        "sleep",
        "patterns",
        "lifestyle",
        "mendeley",
        "y74tv2jbz3"
    ]

    candidate_files = []

    for downloads_folder in possible_download_folders:
        if downloads_folder.exists():
            for file_path in downloads_folder.glob("*"):
                file_name_lower = file_path.name.lower()

                if file_path.suffix.lower() in [".zip", ".csv"]:
                    if any(term in file_name_lower for term in search_terms):
                        candidate_files.append(file_path)

    return sorted(candidate_files, key=lambda path: path.stat().st_mtime, reverse=True)


# ------------------------------------------------------------
# Main loading logic
# ------------------------------------------------------------

csv_files = find_csv_files(BANGLADESH_DATA_DIR)

if len(csv_files) > 0:
    print("\nCSV file already found in project folder. Skipping download.")

else:
    print("\nNo CSV found in project folder yet.")

    # Try automatic Mendeley download first
    try:
        print("Trying automatic download from Mendeley...")
        try_mendeley_download()

    except Exception as error:
        print("\nAutomatic Mendeley download did not work.")
        print(f"Error type: {type(error).__name__}")
        print(f"Error message: {error}")

        print("\nTrying to find the dataset in your Downloads folder...")

        candidate_files = search_downloads_folder()

        if len(candidate_files) == 0:
            print("\nNo matching ZIP or CSV file found in Downloads.")

            print("\nManual fallback:")
            print("1. Open this dataset page:")
            print(DATASET_PAGE_URL)
            print("2. Click Download All.")
            print("3. Run this cell again.")
            print("The notebook will then search your Downloads folder and copy the file automatically.")

        else:
            selected_file = candidate_files[0]

            print("\nFound possible dataset file:")
            print(selected_file)

            destination_path = BANGLADESH_DATA_DIR / selected_file.name

            shutil.copy2(selected_file, destination_path)

            print("\nCopied file into project folder:")
            print(destination_path)

            if destination_path.suffix.lower() == ".zip":
                extract_zip_file(destination_path, BANGLADESH_DATA_DIR)


# ------------------------------------------------------------
# Find and load CSV file
# ------------------------------------------------------------

csv_files = find_csv_files(BANGLADESH_DATA_DIR)

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file was found yet.\n"
        "Download the Mendeley ZIP once, then run this cell again."
    )

bangladesh_csv_path = csv_files[0]

print("\nUsing CSV file:")
print(bangladesh_csv_path)

bangladesh_sleep_raw = pd.read_csv(bangladesh_csv_path)

print(
    f"\nDataset shape: "
    f"{bangladesh_sleep_raw.shape[0]} rows × "
    f"{bangladesh_sleep_raw.shape[1]} columns"
)

bangladesh_sleep_raw.head()

Bangladesh dataset folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\external_sleep_datasets\bangladesh_sleep_patterns

No CSV found in project folder yet.
Trying automatic download from Mendeley...

Automatic Mendeley download did not work.
Error type: HTTPError
Error message: 403 Client Error: Forbidden for url: https://data.mendeley.com/datasets/y74tv2jbz3/1

Trying to find the dataset in your Downloads folder...

Found possible dataset file:
D:\Downloads\Survey Dataset on Sleep Patterns, Health Effects,.zip

Copied file into project folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\external_sleep_datasets\bangladesh_sleep_patterns\Survey Dataset on Sleep Patterns, Health Effects,.zip
Extracted ZIP file: Survey Dataset on Sleep Patterns, Health Effects,.zip

Using CSV file:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\external_sleep_datasets\bangladesh_sleep_patterns\Survey Dataset on Sleep Patterns, Health Effe

,Your Age,What is your weight,Your Height,What is your gender?,What is your occupation?,What time do you usually go to bed?,What time do you usually wake up on working days?,What time do you usually go to bed on weekends?,What time do you usually wake up on weekends?,How long does it take you to fall asleep after going to bed?,How many hours of sleep do you get on average per night?,What are the main reasons you sleep late?,Do you have difficulty falling asleep?,Do you experience breathing difficulties while sleeping,Do you experience restless legs or involuntary movements during sleep?,Do you have any medical conditions that might affect your sleep?,Do you experience any of the following side effects from late sleeping?,How often do you find it hard to concentrate due to lack of sleep?,What strategies do you use to cope with the side effects of late sleeping?,How would you rate the comfort of your sleeping environment
0,18-30,83.0,5.1,Male,Student,10 PM - 12 AM,6 AM - 8 AM,12 AM - 2 AM,8 AM - 10 AM,15-30 minutes,6-8 hours,Study & Exam Purpose;Social Media/Internet;Ent...,Rarely,Never,Sometimes,no,Fatigue;Decreased academic or work performance...,Sometimes,Caffeine or other stimulants;Exercise;Others,4
1,18-30,68.0,5.8,Male,Student,After 2 AM,8 AM - 10 AM,After 2 AM,After 10 AM,15-30 minutes,4-6 hours,Study & Exam Purpose;Social Media/Internet;Ent...,Rarely,Never,Never,Asthma,Difficulty concentrating;Mood swings or irrita...,Sometimes,Naps during the day;Caffeine or other stimulan...,4
2,18-30,50.0,5.8,Male,Student,12 AM - 2 AM,8 AM - 10 AM,12 AM - 2 AM,8 AM - 10 AM,15-30 minutes,6-8 hours,professional Work;Study & Exam Purpose;Social ...,Never,Never,Rarely,No,Fatigue;Mood swings or irritability;Decreased ...,Never,Naps during the day;Exercise,4
3,18-30,70.0,5.5,Female,Student,12 AM - 2 AM,6 AM - 8 AM,After 2 AM,After 10 AM,More than 60 minutes,4-6 hours,"Social Media/Internet;Entertainment (TV, movie...",Always,Sometimes,Sometimes,No,Fatigue;Difficulty concentrating;Mood swings o...,Always,Exercise,3
4,18-30,70.0,5.6,Male,Student,12 AM - 2 AM,6 AM - 8 AM,12 AM - 2 AM,8 AM - 10 AM,Less than 15 minutes,6-8 hours,Study & Exam Purpose;Social Media/Internet;Str...,Sometimes,Rarely,Never,No,Fatigue;Difficulty concentrating;Mood swings o...,Often,Naps during the day;Exercise;Others,4
